# Exercise 6 — Convert RNN-T → TDT (Colab GPU)

Add duration outputs to the joint, fine-tune as a Token-and-Duration Transducer, measure the ~2.7× decoder speedup, and extract word-level timestamps. Needs your Exercise 5 `.nemo` checkpoint.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwkendall/parakeet-study/blob/main/exercises/06-convert-to-tdt/explore.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/cwkendall/parakeet-study/main?labpath=exercises%2F06-convert-to-tdt%2Fexplore.ipynb)

### Step 0 — confirm a GPU is attached

In Colab: **Runtime → Change runtime type → GPU**.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
!nvidia-smi -L

### Step 1 — clone the repo and enter the lab

The scripts you run below are the full reference solution.

In [ ]:
![ -d parakeet-study ] || git clone https://github.com/cwkendall/parakeet-study.git
%cd parakeet-study/exercises/06-convert-to-tdt

### Step 2 — install dependencies (a few minutes)

In [ ]:
!pip -q install -r requirements.txt

### Step 3 — initialise a TDT model from your Exercise 5 RNN-T checkpoint

This adds the extra duration outputs to the joint network.

In [ ]:
!python init_from_rnnt.py \
  --rnnt_model /path/to/ex5_rnnt.nemo \
  --out ./tdt_init.nemo

### Step 4 — fine-tune (smoke test here, overnight for real)

In [ ]:
!python train.py \
  --config-path ./conf --config-name fastconformer_tdt_small \
  +init_from_nemo_model=./tdt_init.nemo trainer.max_steps=50

### Step 5 — the punchline: TDT decoding speedup vs RNN-T

In [ ]:
!python benchmark_tdt.py --model ./nemo_experiments/**/checkpoints/*.nemo --manifest ./data/test-clean.json

### Step 6 — word-level timestamps fall out of the durations

In [ ]:
!python extract_timestamps.py --model ./nemo_experiments/**/checkpoints/*.nemo --audio ./data/sample.wav || echo 'point --audio at any 16 kHz wav'

---

When the smoke test passes, scale up the data and `max_steps` for a real run. The READMEs have the target metrics and reflection questions.